In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd

file_path = "/Volumes/workspace/default/asgairlines_data/UseCase - Airlines.xlsx"

excel_data = pd.read_excel(file_path,sheet_name=None)
print(list(excel_data.keys()))

['flights', 'payments', 'bookings', 'passengers']


In [0]:
flights = excel_data["flights"]
bookings = excel_data["bookings"]
passengers = excel_data["passengers"]
payments = excel_data["payments"]

print("Flights", flights.shape)
print("Bookings", bookings.shape)
print("Passengers", passengers.shape)
print("Payments", payments.shape)

Flights (1020, 7)
Bookings (1000, 9)
Passengers (1039, 9)
Payments (1000, 4)


In [0]:
# RAW DATA PROFILING 

datasets = {"Flights": flights,"Bookings": bookings,"Passengers": passengers,"Payments": payments}
for name, df in datasets.items():
    print("\n" + "-" * 130)
    print(name)
    print("-" * 130)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes.to_string())

    print("\nMissing Values:")
    print(df.isnull().sum().to_string())

    print("\nDuplicate Rows:", df.duplicated().sum())

    print("\nFirst 5 Rows:")
    print(df.head().to_string(index=False))


----------------------------------------------------------------------------------------------------------------------------------
Flights
----------------------------------------------------------------------------------------------------------------------------------

Columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']

Data Types:
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object

Missing Values:
flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0

Duplicate Rows: 15

First 5 Rows:
flight_id   airline source destination          departure_time            arrival_time duration
    SJ010  SpiceJet    CCU         MAA 2026-04-20 23:38:41.701 2026-04-21 02:32:41.7

In [0]:


# FLIGHT ID VALIDATION

flight_id_pattern = r"^[A-Z0-9]{2}\d{3}$"

invalid_flight_ids = flights[
    ~flights["flight_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.match(flight_id_pattern, na=False)
]

print("Total flight records:", len(flights))
print("Invalid flight IDs:", len(invalid_flight_ids))

if len(invalid_flight_ids) > 0:
    print("\nInvalid Flight IDs:")
    print(
        invalid_flight_ids[
            ["flight_id", "airline", "source", "destination"]
        ].to_string(index=False)
    )
else:
    print("All flight IDs passed the validation rule.")

Total flight records: 1020
Invalid flight IDs: 0
All flight IDs passed the validation rule.


In [0]:

# FLIGHT TIME & DURATION QUALITY CHECK


from datetime import time, timedelta, datetime
def convert_duration(value):

    if pd.isna(value):
        return pd.NaT

    # Excel time value
    if isinstance(value, time):
        return pd.Timedelta(
            hours=value.hour,
            minutes=value.minute,
            seconds=value.second,
            microseconds=value.microsecond
        )

    # Python timedelta
    if isinstance(value, timedelta):
        return pd.Timedelta(value)

    # Pandas timestamp/datetime.
    if isinstance(value, (pd.Timestamp, datetime)):
        return pd.Timedelta(
            hours=value.hour,
            minutes=value.minute,
            seconds=value.second,
            microseconds=value.microsecond
        )

   
    return pd.to_timedelta(
        str(value),
        errors="coerce"
    )


# Calculate duration from timestamps
flights["calculated_duration"] = (
    flights["arrival_time"] -
    flights["departure_time"]
)

# Convert provided Excel duration
flights["provided_duration"] = (
    flights["duration"].apply(convert_duration)
)

# Initial duration difference
flights["duration_difference"] = (
    flights["provided_duration"] -
    flights["calculated_duration"]
)

# Identify overnight flights
overnight_flights_initial = flights[
    flights["arrival_time"].dt.date >
    flights["departure_time"].dt.date
]

# Identify negative durations
negative_duration_initial = flights[
    flights["calculated_duration"] < pd.Timedelta(0)
]

# Identify duration mismatches
duration_mismatch_initial = flights[
    flights["provided_duration"].notna() &
    (
        flights["provided_duration"] !=
        flights["calculated_duration"]
    )
]

print("Initial overnight flights:", len(overnight_flights_initial))
print(
    "Initial negative calculated durations:",
    len(negative_duration_initial)
)
print(
    "Initial duration mismatches:",
    len(duration_mismatch_initial)
)

Initial overnight flights: 124
Initial negative calculated durations: 1
Initial duration mismatches: 1


In [0]:

# CORRECT FLIGHT TIME ANOMALIES

duration_corrections = 0

negative_mask = (
    flights["calculated_duration"] <
    pd.Timedelta(0)
)

for index in flights.index[negative_mask]:

    provided_duration = flights.loc[
        index,
        "provided_duration"
    ]

    # the arrival timestamp.
    if pd.notna(provided_duration):

        flights.loc[index, "arrival_time"] = (
            flights.loc[index, "departure_time"] +
            provided_duration
        )

        duration_corrections += 1


# Recalculate durations after correction
flights["calculated_duration"] = (
    flights["arrival_time"] -
    flights["departure_time"]
)

flights["duration_difference"] = (
    flights["provided_duration"] -
    flights["calculated_duration"]
)

print("Flight duration corrections applied:",
      duration_corrections)

Flight duration corrections applied: 1


In [0]:

# VERIFY FLIGHT TIME CORRECTION

sj192 = flights[
    flights["flight_id"].astype(str).str.upper() == "SJ192"
]

print(
    sj192[
        [
            "flight_id",
            "departure_time",
            "arrival_time",
            "duration",
            "provided_duration",
            "calculated_duration",
            "duration_difference"
        ]
    ].to_string(index=False)
)

flight_id      departure_time        arrival_time            duration provided_duration calculated_duration duration_difference
    SJ192 2026-04-19 18:45:42 2026-04-19 23:45:42 1899-12-29 05:00:00   0 days 05:00:00     0 days 05:00:00              0 days


In [0]:

# BOOKINGS to FLIGHTS

flight_ids = set(
    flights["flight_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

booking_flight_ids = (
    bookings["flight_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

invalid_booking_flights = bookings[
    ~booking_flight_ids.isin(flight_ids)
]

print(
    "Bookings with invalid flight_id:",
    len(invalid_booking_flights)
)

if len(invalid_booking_flights) == 0:
    print("All booking flight IDs exist in Flights.")
else:
    print(
        invalid_booking_flights[
            ["booking_id", "flight_id"]
        ].to_string(index=False)
    )

Bookings with invalid flight_id: 0
All booking flight IDs exist in Flights.


In [0]:

# BOOKINGS to PASSENGERS

passenger_ids = set(
    passengers["passenger_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

booking_passenger_ids = (
    bookings["passenger_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

invalid_booking_passengers = bookings[
    ~booking_passenger_ids.isin(passenger_ids)
]

print("Total bookings:", len(bookings))
print(
    "Bookings with invalid passenger_id:",
    len(invalid_booking_passengers)
)

if len(invalid_booking_passengers) > 0:
    print("\nInvalid passenger references:")
    print(
        invalid_booking_passengers[
            ["booking_id", "passenger_id"]
        ].to_string(index=False)
    )
else:
    print("All booking passenger IDs exist in Passengers.")

Total bookings: 1000
Bookings with invalid passenger_id: 0
All booking passenger IDs exist in Passengers.


In [0]:

# FLIGHTS — CLEANING

flights_clean = flights.copy()

for col in [
    "flight_id",
    "airline",
    "source",
    "destination"
]:
    flights_clean[col] = (
        flights_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Fill missing airline values

airline_mapping = {
    "AI": "AIR INDIA",
    "SJ": "SPICEJET",
    "UK": "VISTARA",
    "6F": "AIRLINE 6F"
}


def infer_airline(flight_id):

    if pd.isna(flight_id):
        return "UNKNOWN"

    prefix = str(flight_id)[:2]

    return airline_mapping.get(
        prefix,
        "UNKNOWN"
    )


flights_clean["airline"] = (
    flights_clean.apply(
        lambda row:
        infer_airline(row["flight_id"])
        if pd.isna(row["airline"])
        else row["airline"],
        axis=1
    )
)

# Remove exact duplicate rows

before_flight_rows = len(flights_clean)

flights_clean = (
    flights_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

exact_duplicates_removed = (
    before_flight_rows -
    len(flights_clean)
)


# Remove duplicate flight IDs

duplicate_flight_ids = (
    flights_clean[
        flights_clean["flight_id"].duplicated(
            keep=False
        )
    ]["flight_id"]
    .nunique()
)

print(
    "Duplicate flight IDs before business-key deduplication:",
    duplicate_flight_ids
)

# Sort so that earliest departure is retained
flights_clean = (
    flights_clean
    .sort_values(
        by=["flight_id", "departure_time"]
    )
    .drop_duplicates(
        subset=["flight_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

# Total flight rows removed

flight_duplicates_removed = (
    before_flight_rows -
    len(flights_clean)
)


# Duration in minutes

flights_clean["duration_minutes"] = (
    flights_clean["calculated_duration"]
    .dt.total_seconds() / 60
)


# Validation output

print(
    "Flights before cleaning:",
    before_flight_rows
)

print(
    "Exact duplicate rows removed:",
    exact_duplicates_removed
)

print(
    "Duplicate flight IDs handled:",
    duplicate_flight_ids
)

print(
    "Total flight rows removed:",
    flight_duplicates_removed
)

print(
    "Flights after cleaning:",
    len(flights_clean)
)

print(
    "Missing airlines:",
    flights_clean["airline"].isna().sum()
)

print(
    "Remaining duplicate flight IDs:",
    flights_clean["flight_id"]
    .duplicated()
    .sum()
)

Duplicate flight IDs before business-key deduplication: 1
Flights before cleaning: 1020
Exact duplicate rows removed: 15
Duplicate flight IDs handled: 1
Total flight rows removed: 16
Flights after cleaning: 1004
Missing airlines: 0
Remaining duplicate flight IDs: 0


In [0]:

# FLIGHT QUALITY FLAGS
# Recalculate after cleaning/correction
flights_clean["calculated_duration"] = (
    flights_clean["arrival_time"] -
    flights_clean["departure_time"]
)

flights_clean["duration_minutes"] = (
    flights_clean["calculated_duration"]
    .dt.total_seconds() / 60
)

# Overnight indicator
flights_clean["is_overnight"] = (
    flights_clean["arrival_time"].dt.date >
    flights_clean["departure_time"].dt.date
)

# Extreme duration anomaly
flights_clean["duration_anomaly"] = (
    flights_clean["duration_minutes"] > 600
)

# Duration correction indicator
flights_clean["duration_corrected"] = (
    flights_clean["flight_id"] == "SJ192"
)

print("Final overnight flights:",
      flights_clean["is_overnight"].sum())

print(
    "Extreme duration anomalies:",
    flights_clean["duration_anomaly"].sum()
)

print(
    "Duration corrections:",
    flights_clean["duration_corrected"].sum()
)

Final overnight flights: 122
Extreme duration anomalies: 0
Duration corrections: 1


In [0]:

# BOOKINGS — CLEANING

bookings_clean = bookings.copy()

text_columns = [
    "booking_id",
    "passenger_id",
    "flight_id",
    "status",
    "passport_number",
    "seat_number",
    "emergency_contact_name",
    "emergency_contact_phone"
]

for col in text_columns:

    bookings_clean[col] = (
        bookings_clean[col]
        .astype("string")
        .str.strip()
    )


# Standardize status
bookings_clean["status"] = (
    bookings_clean["status"]
    .str.upper()
    .fillna("UNKNOWN")
)


# converted to a valid booking status.
bookings_clean["status"] = (
    bookings_clean["status"]
    .replace("INVALID", "UNKNOWN")
)


# Remove exact duplicates
before_booking_rows = len(bookings_clean)

bookings_clean = (
    bookings_clean
    .drop_duplicates()
    .reset_index(drop=True)
)


print("Bookings before cleaning:",
      before_booking_rows)

print(
    "Duplicate rows removed:",
    before_booking_rows - len(bookings_clean)
)

print("Bookings after cleaning:",
      len(bookings_clean))

print("\nFinal booking status distribution:")
print(
    bookings_clean["status"]
    .value_counts()
    .to_string()
)


Bookings before cleaning: 1000
Duplicate rows removed: 0
Bookings after cleaning: 1000

Final booking status distribution:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       75


In [0]:

# PASSENGERS - CLEANING

passengers_clean = passengers.copy()

# Standardize text columns
text_columns = [
    "passenger_id",
    "first_name",
    "last_name",
    "gender",
    "email",
    "phone"
]

for col in text_columns:
    passengers_clean[col] = (
        passengers_clean[col]
        .astype("string")
        .str.strip()
    )

# Standardize gender
passengers_clean["gender"] = (
    passengers_clean["gender"]
    .str.upper()
)

# Handle missing last names
passengers_clean["last_name"] = (
    passengers_clean["last_name"]
    .fillna("UNKNOWN")
)

# Validate age
invalid_age = passengers_clean[
    (passengers_clean["age"] < 0) |
    (passengers_clean["age"] > 120)
]

# Remove exact duplicates as a safety check
before = len(passengers_clean)

passengers_clean = passengers_clean.drop_duplicates()

duplicates_removed = before - len(passengers_clean)

print("Passengers before cleaning:", before)
print("Duplicate rows removed:", duplicates_removed)
print("Passengers after cleaning:", len(passengers_clean))

print("\nMissing last names after cleaning:",
      passengers_clean["last_name"].isna().sum())

print("\nInvalid ages:", len(invalid_age))

print("\nAge range:",
      passengers_clean["age"].min(),
      "to",
      passengers_clean["age"].max())

print("\nGender distribution:")
print(
    passengers_clean["gender"]
    .value_counts(dropna=False)
    .to_string()
)

Passengers before cleaning: 1039
Duplicate rows removed: 0
Passengers after cleaning: 1039

Missing last names after cleaning: 0

Invalid ages: 0

Age range: 1 to 89

Gender distribution:
gender
M    545
F    494


In [0]:

# PII MASKING / HASHING
import hashlib
def sha256_hash(value):

    if pd.isna(value):
        return None

    return hashlib.sha256(
        str(value)
        .encode("utf-8")
    ).hexdigest()


def mask_name(value):

    if pd.isna(value):
        return "UNKNOWN"

    text = str(value).strip()

    if text == "":
        return "UNKNOWN"

    if len(text) <= 2:
        return "*" * len(text)

    return (
        text[0] +
        "*" * (len(text) - 2) +
        text[-1]
    )


passengers_safe = passengers_clean.copy()


# Hash sensitive identifiers
passengers_safe["passenger_id_hash"] = (
    passengers_safe["passenger_id"]
    .apply(sha256_hash)
)

passengers_safe["email_hash"] = (
    passengers_safe["email"]
    .apply(sha256_hash)
)

passengers_safe["phone_hash"] = (
    passengers_safe["phone"]
    .apply(sha256_hash)
)

passengers_safe["aadhaar_hash"] = (
    passengers_safe["aadhaar_id"]
    .apply(sha256_hash)
)


# Mask names
passengers_safe["first_name_masked"] = (
    passengers_safe["first_name"]
    .apply(mask_name)
)

passengers_safe["last_name_masked"] = (
    passengers_safe["last_name"]
    .apply(mask_name)
)


# Keep only safe analytical fields
passengers_safe["birth_year"] = (
    passengers_safe["date_of_birth"]
    .dt.year
)


passengers_safe = passengers_safe[
    [
        "passenger_id_hash",
        "first_name_masked",
        "last_name_masked",
        "age",
        "gender",
        "email_hash",
        "phone_hash",
        "aadhaar_hash",
        "birth_year"
    ]
].copy()


print(
    "PII-safe passenger dataset created."
)

print(
    "Rows:",
    len(passengers_safe)
)

print(
    "Columns:",
    len(passengers_safe.columns)
)

print("\nColumns:")
print(passengers_safe.columns.tolist())

PII-safe passenger dataset created.
Rows: 1039
Columns: 9

Columns:
['passenger_id_hash', 'first_name_masked', 'last_name_masked', 'age', 'gender', 'email_hash', 'phone_hash', 'aadhaar_hash', 'birth_year']


In [0]:

# BOOKINGS — PII SAFE VERSION

bookings_safe = bookings_clean.copy()


# Hash passport number
bookings_safe["passport_hash"] = (
    bookings_safe["passport_number"]
    .apply(sha256_hash)
)


# Hash emergency phone
bookings_safe["emergency_contact_phone_hash"] = (
    bookings_safe["emergency_contact_phone"]
    .apply(sha256_hash)
)


# Mask emergency contact name
bookings_safe["emergency_contact_name_masked"] = (
    bookings_safe["emergency_contact_name"]
    .apply(mask_name)
)


# Keep safe columns
bookings_safe = bookings_safe[
    [
        "booking_id",
        "passenger_id",
        "flight_id",
        "booking_date",
        "status",
        "seat_number",
        "passport_hash",
        "emergency_contact_name_masked",
        "emergency_contact_phone_hash"
    ]
].copy()


print("PII-safe bookings created.")
print("Rows:", len(bookings_safe))
print("Columns:", len(bookings_safe.columns))

PII-safe bookings created.
Rows: 1000
Columns: 9


In [0]:

# PAYMENTS — CLEANING
payments_clean = payments.copy()


# Standardize text columns
for col in [
    "payment_id",
    "booking_id",
    "payment_method"
]:

    payments_clean[col] = (
        payments_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )


# Convert invalid strings and missing values into NULL
payments_clean["amount"] = pd.to_numeric(
    payments_clean["amount"],
    errors="coerce"
)


# Round monetary values
payments_clean["amount"] = (
    payments_clean["amount"]
    .round(2)
)


# Remove exact duplicates
before_payment_rows = len(payments_clean)

payments_clean = (
    payments_clean
    .drop_duplicates()
    .reset_index(drop=True)
)


print("Payments before cleaning:",
      before_payment_rows)

print(
    "Duplicate rows removed:",
    before_payment_rows - len(payments_clean)
)

print("Payments after cleaning:",
      len(payments_clean))

print(
    "Valid payment amounts:",
    payments_clean["amount"].notna().sum()
)

print(
    "Missing/invalid amounts:",
    payments_clean["amount"].isna().sum()
)

print(
    "Negative amounts:",
    (payments_clean["amount"] < 0).sum()
)

Payments before cleaning: 1000
Duplicate rows removed: 0
Payments after cleaning: 1000
Valid payment amounts: 922
Missing/invalid amounts: 78
Negative amounts: 0


In [0]:

# FINAL SILVER DATASETS

flights_silver = flights_clean[
    [
        "flight_id",
        "airline",
        "source",
        "destination",
        "departure_time",
        "arrival_time",
        "duration_minutes",
        "is_overnight",
        "duration_anomaly",
        "duration_corrected"
    ]
].copy()

# Bookings Silver

bookings_silver = bookings_safe.copy()


# Passengers Silver

passengers_silver = passengers_safe.copy()


# Payments Silver

payments_silver = payments_clean[
    [
        "payment_id",
        "booking_id",
        "amount",
        "payment_method"
    ]
].copy()


print("Silver datasets created.")

print("\nFlights Silver:", flights_silver.shape)
print("Bookings Silver:", bookings_silver.shape)
print("Passengers Silver:", passengers_silver.shape)
print("Payments Silver:", payments_silver.shape)

Silver datasets created.

Flights Silver: (1004, 10)
Bookings Silver: (1000, 9)
Passengers Silver: (1039, 9)
Payments Silver: (1000, 4)


In [0]:

# FINAL SILVER VALIDATION

print("=" * 70)
print("FINAL SILVER DATA QUALITY VALIDATION")
print("=" * 70)


# Flights
print("\nFLIGHTS")
print("-" * 40)

print("Rows:", len(flights_silver))

print(
    "Missing airline:",
    flights_silver["airline"].isna().sum()
)

print(
    "Duplicate rows:",
    flights_silver.duplicated().sum()
)

print(
    "Negative durations:",
    (
        flights_silver["duration_minutes"] < 0
    ).sum()
)

print(
    "Missing durations:",
    flights_silver["duration_minutes"].isna().sum()
)

print(
    "Invalid flight IDs:",
    len(
        flights_silver[
            ~flights_silver["flight_id"]
            .str.match(
                flight_id_pattern,
                na=False
            )
        ]
    )
)


# Bookings
print("\nBOOKINGS")
print("-" * 40)

print("Rows:", len(bookings_silver))

print(
    "Missing status:",
    bookings_silver["status"].isna().sum()
)

print(
    "INVALID status:",
    (
        bookings_silver["status"] == "INVALID"
    ).sum()
)

print(
    "Duplicate rows:",
    bookings_silver.duplicated().sum()
)


# Passengers
print("\nPASSENGERS")
print("-" * 40)

print("Rows:", len(passengers_silver))

print(
    "Duplicate rows:",
    passengers_silver.duplicated().sum()
)

print(
    "Invalid ages:",
    (
        (passengers_clean["age"] < 0) |
        (passengers_clean["age"] > 120)
    ).sum()
)


# Payments
print("\nPAYMENTS")
print("-" * 40)

print("Rows:", len(payments_silver))

print(
    "Valid amounts:",
    payments_silver["amount"].notna().sum()
)

print(
    "Missing/invalid amounts:",
    payments_silver["amount"].isna().sum()
)

print(
    "Negative amounts:",
    (payments_silver["amount"] < 0).sum()
)

print(
    "Duplicate rows:",
    payments_silver.duplicated().sum()
)


print("\n" + "=" * 70)
print("SILVER VALIDATION COMPLETE")
print("=" * 70)

FINAL SILVER DATA QUALITY VALIDATION

FLIGHTS
----------------------------------------
Rows: 1004
Missing airline: 0
Duplicate rows: 0
Negative durations: 0
Missing durations: 0
Invalid flight IDs: 0

BOOKINGS
----------------------------------------
Rows: 1000
Missing status: 0
INVALID status: 0
Duplicate rows: 0

PASSENGERS
----------------------------------------
Rows: 1039
Duplicate rows: 0
Invalid ages: 0

PAYMENTS
----------------------------------------
Rows: 1000
Valid amounts: 922
Missing/invalid amounts: 78
Negative amounts: 0
Duplicate rows: 0

SILVER VALIDATION COMPLETE


In [0]:

# GOLD — FACT FLIGHTS

fact_flights = flights_silver.copy()


# Date information
fact_flights["departure_date"] = (
    fact_flights["departure_time"]
    .dt.date
)


fact_flights["departure_hour"] = (
    fact_flights["departure_time"]
    .dt.hour
)

# Route
fact_flights["route"] = (
    fact_flights["source"] +
    " → " +
    fact_flights["destination"]
)


print("Fact Flights created.")

print("Rows:", len(fact_flights))

print("Columns:")
print(fact_flights.columns.tolist())

Fact Flights created.
Rows: 1004
Columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration_minutes', 'is_overnight', 'duration_anomaly', 'duration_corrected', 'departure_date', 'departure_hour', 'route']


In [0]:

# GOLD — DIMENSION ROUTE
dim_route = (
    fact_flights
    .groupby(
        [
            "source",
            "destination",
            "route"
        ],
        as_index=False
    )
    .agg(
        total_flights=(
            "flight_id",
            "count"
        ),
        avg_duration_minutes=(
            "duration_minutes",
            "mean"
        ),
        overnight_flights=(
            "is_overnight",
            "sum"
        )
    )
)


dim_route["avg_duration_minutes"] = (
    dim_route["avg_duration_minutes"]
    .round(2)
)


dim_route["route_rank"] = (
    dim_route["total_flights"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)


print("Route dimension created.")

print("Unique routes:", len(dim_route))

print(
    dim_route
    .sort_values(
        "total_flights",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

Route dimension created.
Unique routes: 30
source destination     route  total_flights  avg_duration_minutes  overnight_flights  route_rank
   BOM         CCU BOM → CCU             90                169.51                 16           1
   CCU         DEL CCU → DEL             72                153.61                 11           2
   MAA         BLR MAA → BLR             65                172.82                  8           3
   BLR         BOM BLR → BOM             60                147.73                  7           4
   HYD         MAA HYD → MAA             57                152.81                  5           5
   DEL         HYD DEL → HYD             54                174.76                  6           6
   HYD         DEL HYD → DEL             42                185.36                  9           7
   BOM         DEL BOM → DEL             39                153.82                  3           8
   CCU         BOM CCU → BOM             33                163.97                  1

In [0]:

# GOLD — DIMENSION AIRLINE

dim_airline = (
    fact_flights
    .groupby(
        ["airline"],
        as_index=False
    )
    .agg(
        total_flights=(
            "flight_id",
            "count"
        ),
        avg_duration_minutes=(
            "duration_minutes",
            "mean"
        ),
        overnight_flights=(
            "is_overnight",
            "sum"
        )
    )
)


dim_airline["avg_duration_minutes"] = (
    dim_airline["avg_duration_minutes"]
    .round(2)
)


dim_airline["airline_rank"] = (
    dim_airline["total_flights"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)


print("Airline dimension created.")

print(
    dim_airline
    .sort_values(
        "total_flights",
        ascending=False
    )
    .to_string(index=False)
)

Airline dimension created.
   airline  total_flights  avg_duration_minutes  overnight_flights  airline_rank
    INDIGO            249                167.70                 30             1
 AIR INDIA            247                165.17                 31             2
  SPICEJET            241                163.64                 28             3
   VISTARA            226                163.67                 27             4
   UNKNOWN             29                158.72                  4             5
AIRLINE 6F             12                134.50                  2             6


In [0]:

# GOLD — DIMENSION DATE

all_dates = pd.concat(
    [
        fact_flights["departure_time"].dt.date,
        bookings_clean["booking_date"].dt.date
    ]
)

min_date = min(all_dates)
max_date = max(all_dates)

date_range = pd.date_range(
    start=min_date,
    end=max_date,
    freq="D"
)


dim_date = pd.DataFrame({
    "date": date_range
})


dim_date["year"] = (
    dim_date["date"].dt.year
)

dim_date["month"] = (
    dim_date["date"].dt.month
)

dim_date["month_name"] = (
    dim_date["date"].dt.month_name()
)

dim_date["quarter"] = (
    "Q" +
    dim_date["date"]
    .dt.quarter
    .astype(str)
)

dim_date["day"] = (
    dim_date["date"].dt.day
)

dim_date["day_name"] = (
    dim_date["date"].dt.day_name()
)


print("Date dimension created.")

print("Rows:", len(dim_date))

print(
    dim_date.head().to_string(index=False)
)

Date dimension created.
Rows: 369
      date  year  month month_name quarter  day day_name
2025-04-17  2025      4      April      Q2   17 Thursday
2025-04-18  2025      4      April      Q2   18   Friday
2025-04-19  2025      4      April      Q2   19 Saturday
2025-04-20  2025      4      April      Q2   20   Sunday
2025-04-21  2025      4      April      Q2   21   Monday


In [0]:

# GOLD — DIMENSION PASSENGER

dim_passenger = passengers_silver.copy()

print("Passenger dimension created.")

print("Rows:", len(dim_passenger))

print("Columns:")
print(dim_passenger.columns.tolist())

Passenger dimension created.
Rows: 1039
Columns:
['passenger_id_hash', 'first_name_masked', 'last_name_masked', 'age', 'gender', 'email_hash', 'phone_hash', 'aadhaar_hash', 'birth_year']


In [0]:

# GOLD — FACT BOOKINGS
# Create unique passenger key mapping


passenger_key_map = (
    passengers_clean[
        ["passenger_id"]
    ]
    .drop_duplicates(
        subset=["passenger_id"],
        keep="first"
    )
    .copy()
)


# Create hashed passenger ID
passenger_key_map["passenger_id_hash"] = (
    passenger_key_map["passenger_id"]
    .apply(sha256_hash)
)

# Create Fact Bookings

fact_bookings = bookings_silver.copy()


# Attach hashed passenger key

fact_bookings = fact_bookings.merge(
    passenger_key_map,
    on="passenger_id",
    how="left",
    validate="many_to_one"
)

# Remove raw passenger ID

fact_bookings = fact_bookings.drop(
    columns=["passenger_id"]
)


# Booking date attributes

fact_bookings["booking_year"] = (
    fact_bookings["booking_date"]
    .dt.year
)

fact_bookings["booking_month"] = (
    fact_bookings["booking_date"]
    .dt.month
)

fact_bookings["booking_month_name"] = (
    fact_bookings["booking_date"]
    .dt.month_name()
)


# Booking status flags

fact_bookings["is_confirmed"] = (
    fact_bookings["status"] == "CONFIRMED"
)

fact_bookings["is_cancelled"] = (
    fact_bookings["status"] == "CANCELLED"
)

fact_bookings["is_pending"] = (
    fact_bookings["status"] == "PENDING"
)

# Reorder columns
fact_bookings = fact_bookings[
    [
        "booking_id",
        "passenger_id_hash",
        "flight_id",
        "booking_date",
        "booking_year",
        "booking_month",
        "booking_month_name",
        "status",
        "is_confirmed",
        "is_cancelled",
        "is_pending",
        "seat_number",
        "passport_hash",
        "emergency_contact_name_masked",
        "emergency_contact_phone_hash"
    ]
].copy()


# Final validation

print("Fact Bookings created.")

print(
    "Rows:",
    len(fact_bookings)
)

print(
    "Unique booking IDs:",
    fact_bookings["booking_id"].nunique()
)

print(
    "Duplicate booking IDs:",
    fact_bookings["booking_id"]
    .duplicated()
    .sum()
)

print(
    "Missing hashed passenger IDs:",
    fact_bookings["passenger_id_hash"]
    .isna()
    .sum()
)

print(
    "Confirmed:",
    fact_bookings["is_confirmed"].sum()
)

print(
    "Cancelled:",
    fact_bookings["is_cancelled"].sum()
)

print(
    "Pending:",
    fact_bookings["is_pending"].sum()
)

print(
    "Unknown:",
    (
        fact_bookings["status"] == "UNKNOWN"
    ).sum()
)

Fact Bookings created.
Rows: 1000
Unique booking IDs: 1000
Duplicate booking IDs: 0
Missing hashed passenger IDs: 0
Confirmed: 320
Cancelled: 314
Pending: 291
Unknown: 75


In [0]:

# DATA QUALITY CHECK - FLIGHT ID UNIQUENESS


duplicate_flight_ids = flights_silver[
    flights_silver.duplicated(
        subset=["flight_id"],
        keep=False
    )
].sort_values("flight_id")

print("Duplicate flight_id rows:", len(duplicate_flight_ids))

if len(duplicate_flight_ids) == 0:
    print("PASS: Every flight_id is unique.")
else:
    print("WARNING: Duplicate flight_ids still exist.")
    display(duplicate_flight_ids)

Duplicate flight_id rows: 0
PASS: Every flight_id is unique.


In [0]:

# GOLD — FACT PAYMENTS

fact_payments = payments_silver.copy()


# Valid payment indicator
fact_payments["payment_valid"] = (
    fact_payments["amount"].notna()
)


fact_payments["amount"] = (
    fact_payments["amount"]
    .round(2)
)


print("Fact Payments created.")

print("Rows:", len(fact_payments))

print(
    "Valid payments:",
    fact_payments["payment_valid"].sum()
)

print(
    "Invalid/missing payments:",
    (~fact_payments["payment_valid"]).sum()
)

print("\nPayment method distribution:")
print(
    fact_payments["payment_method"]
    .value_counts()
    .to_string()
)

Fact Payments created.
Rows: 1000
Valid payments: 922
Invalid/missing payments: 78

Payment method distribution:
payment_method
UPI           358
CARD          329
NETBANKING    313


In [0]:

# GOLD — DATA QUALITY / ANOMALY SUMMARY

quality_summary = pd.DataFrame(
    {
        "metric": [
            "invalid_flight_ids",
            "duplicate_flight_rows_removed",
            "duration_corrections",
            "negative_duration_issues_initial",
            "duration_mismatches_initial",
            "extreme_duration_anomalies",
            "overnight_flights",
            "missing_booking_status_resolved",
            "invalid_booking_status_resolved",
            "missing_passenger_last_names_resolved",
            "missing_or_invalid_payment_amounts"
        ],
        "value": [
            len(invalid_flight_ids),
            flight_duplicates_removed,
            duration_corrections,
            len(negative_duration_initial),
            len(duration_mismatch_initial),
            int(
                fact_flights[
                    "duration_anomaly"
                ].sum()
            ),
            int(
                fact_flights[
                    "is_overnight"
                ].sum()
            ),
            int(
                bookings["status"]
                .isna()
                .sum()
            ),
            int(
                (bookings["status"] == "INVALID")
                .sum()
            ),
            int(
                passengers["last_name"]
                .isna()
                .sum()
            ),
            int(
                payments_clean["amount"]
                .isna()
                .sum()
            )
        ]
    }
)


print(
    quality_summary.to_string(index=False)
)

                               metric  value
                   invalid_flight_ids      0
        duplicate_flight_rows_removed     16
                 duration_corrections      1
     negative_duration_issues_initial      1
          duration_mismatches_initial      1
           extreme_duration_anomalies      0
                    overnight_flights    122
      missing_booking_status_resolved     45
      invalid_booking_status_resolved     30
missing_passenger_last_names_resolved     10
   missing_or_invalid_payment_amounts     78


In [0]:

# GOLD — KPI SUMMARY
valid_payment_amounts = fact_payments[
    fact_payments["payment_valid"]
]["amount"]


kpi_summary = {
    "total_flights":
        len(fact_flights),

    "total_routes":
        fact_flights["route"].nunique(),

    "total_airlines":
        fact_flights["airline"].nunique(),

    "average_flight_duration_minutes":
        round(
            fact_flights[
                "duration_minutes"
            ].mean(),
            2
        ),

    "overnight_flights":
        int(
            fact_flights[
                "is_overnight"
            ].sum()
        ),

    "extreme_duration_anomalies":
        int(
            fact_flights[
                "duration_anomaly"
            ].sum()
        ),

    "duration_corrections":
        int(
            fact_flights[
                "duration_corrected"
            ].sum()
        ),

    "total_bookings":
        len(fact_bookings),

    "confirmed_bookings":
        int(
            fact_bookings[
                "is_confirmed"
            ].sum()
        ),

    "cancelled_bookings":
        int(
            fact_bookings[
                "is_cancelled"
            ].sum()
        ),

    "pending_bookings":
        int(
            fact_bookings[
                "is_pending"
            ].sum()
        ),

    "unknown_bookings":
        int(
            (
                fact_bookings["status"] ==
                "UNKNOWN"
            ).sum()
        ),

    "valid_payments":
        int(
            fact_payments[
                "payment_valid"
            ].sum()
        ),

    "invalid_or_missing_payments":
        int(
            (
                ~fact_payments[
                    "payment_valid"
                ]
            ).sum()
        ),

    "total_revenue":
        round(
            valid_payment_amounts.sum(),
            2
        ),

    "average_payment_amount":
        round(
            valid_payment_amounts.mean(),
            2
        )
}


kpi_summary = pd.DataFrame(
    [kpi_summary]
)


print(
    kpi_summary.to_string(index=False)
)

 total_flights  total_routes  total_airlines  average_flight_duration_minutes  overnight_flights  extreme_duration_anomalies  duration_corrections  total_bookings  confirmed_bookings  cancelled_bookings  pending_bookings  unknown_bookings  valid_payments  invalid_or_missing_payments  total_revenue  average_payment_amount
          1004            30               6                           164.54                122                           0                     1            1000                 320                 314               291                75             922                           78     7385142.98                 8009.92


In [0]:

# FINAL GOLD VALIDATION

print("=" * 70)
print("FINAL GOLD VALIDATION")
print("=" * 70)


print("\nFACT FLIGHTS")
print("-" * 40)
print("Rows:", len(fact_flights))
print(
    "Overnight flights:",
    fact_flights["is_overnight"].sum()
)
print(
    "Extreme duration anomalies:",
    fact_flights["duration_anomaly"].sum()
)
print(
    "Duration corrections:",
    fact_flights["duration_corrected"].sum()
)


print("\nDIM ROUTE")
print("-" * 40)
print("Rows:", len(dim_route))


print("\nDIM AIRLINE")
print("-" * 40)
print("Rows:", len(dim_airline))


print("\nDIM DATE")
print("-" * 40)
print("Rows:", len(dim_date))


print("\nDIM PASSENGER")
print("-" * 40)
print("Rows:", len(dim_passenger))


print("\nFACT BOOKINGS")
print("-" * 40)
print("Rows:", len(fact_bookings))
print(
    "Confirmed:",
    fact_bookings["is_confirmed"].sum()
)
print(
    "Cancelled:",
    fact_bookings["is_cancelled"].sum()
)
print(
    "Pending:",
    fact_bookings["is_pending"].sum()
)


print("\nFACT PAYMENTS")
print("-" * 40)
print("Rows:", len(fact_payments))
print(
    "Valid payments:",
    fact_payments["payment_valid"].sum()
)
print(
    "Invalid/missing payments:",
    (~fact_payments["payment_valid"]).sum()
)


print("\nKPI SUMMARY")
print("-" * 40)
print(
    kpi_summary.to_string(index=False)
)


print("\n" + "=" * 70)
print("GOLD VALIDATION COMPLETE")
print("=" * 70)

FINAL GOLD VALIDATION

FACT FLIGHTS
----------------------------------------
Rows: 1004
Overnight flights: 122
Extreme duration anomalies: 0
Duration corrections: 1

DIM ROUTE
----------------------------------------
Rows: 30

DIM AIRLINE
----------------------------------------
Rows: 6

DIM DATE
----------------------------------------
Rows: 369

DIM PASSENGER
----------------------------------------
Rows: 1039

FACT BOOKINGS
----------------------------------------
Rows: 1000
Confirmed: 320
Cancelled: 314
Pending: 291

FACT PAYMENTS
----------------------------------------
Rows: 1000
Valid payments: 922
Invalid/missing payments: 78

KPI SUMMARY
----------------------------------------
 total_flights  total_routes  total_airlines  average_flight_duration_minutes  overnight_flights  extreme_duration_anomalies  duration_corrections  total_bookings  confirmed_bookings  cancelled_bookings  pending_bookings  unknown_bookings  valid_payments  invalid_or_missing_payments  total_revenue  aver

In [0]:

# CREATE STORAGE DIRECTORIES

base_path = (
    "/Volumes/workspace/default/"
    "asgairlines_data"
)

silver_path = base_path + "/clean"
gold_path = base_path + "/gold"


dbutils.fs.mkdirs(silver_path)
dbutils.fs.mkdirs(gold_path)


print("Storage directories ready.")

print("Silver:", silver_path)
print("Gold:", gold_path)

Storage directories ready.
Silver: /Volumes/workspace/default/asgairlines_data/clean
Gold: /Volumes/workspace/default/asgairlines_data/gold


In [0]:

# SAVE SILVER DATASETS

flights_silver.to_csv(
    silver_path + "/flights_silver.csv",
    index=False
)

bookings_silver.to_csv(
    silver_path + "/bookings_silver.csv",
    index=False
)

passengers_silver.to_csv(
    silver_path + "/passengers_silver.csv",
    index=False
)

payments_silver.to_csv(
    silver_path + "/payments_silver.csv",
    index=False
)


print("Silver datasets saved successfully.")

Silver datasets saved successfully.


In [0]:

# SAVE GOLD ANALYTICAL TABLES

fact_flights.to_csv(
    gold_path + "/fact_flights.csv",
    index=False
)

fact_bookings.to_csv(
    gold_path + "/fact_bookings.csv",
    index=False
)

fact_payments.to_csv(
    gold_path + "/fact_payments.csv",
    index=False
)

dim_route.to_csv(
    gold_path + "/dim_route.csv",
    index=False
)

dim_airline.to_csv(
    gold_path + "/dim_airline.csv",
    index=False
)

dim_date.to_csv(
    gold_path + "/dim_date.csv",
    index=False
)

dim_passenger.to_csv(
    gold_path + "/dim_passenger.csv",
    index=False
)

kpi_summary.to_csv(
    gold_path + "/kpi_summary.csv",
    index=False
)

quality_summary.to_csv(
    gold_path + "/quality_summary.csv",
    index=False
)


print("Gold datasets saved successfully.")

Gold datasets saved successfully.


In [0]:

# VERIFY SILVER FILES

print("Files in Silver layer:\n")

for file in dbutils.fs.ls(silver_path):
    print(file.name)

Files in Silver layer:

bookings_silver.csv
flights_silver.csv
passengers_silver.csv
payments_silver.csv


In [0]:

# VERIFY GOLD FILES
# ==========================================

print("Files in Gold layer:\n")

for file in dbutils.fs.ls(gold_path):
    print(file.name)

Files in Gold layer:

dim_airline.csv
dim_date.csv
dim_passenger.csv
dim_route.csv
fact_bookings.csv
fact_flights.csv
fact_payments.csv
kpi_summary.csv
quality_summary.csv


In [0]:

# FINAL PIPELINE SUMMARY

print("=" * 70)
print("ASG AIRLINES DATA ENGINEERING PIPELINE")
print("FINAL SUMMARY")
print("=" * 70)

print("\nRAW DATA")
print("-" * 40)
print("Flights:", len(flights))
print("Bookings:", len(bookings))
print("Passengers:", len(passengers))
print("Payments:", len(payments))


print("\nSILVER DATA")
print("-" * 40)
print("Flights:", len(flights_silver))
print("Bookings:", len(bookings_silver))
print("Passengers:", len(passengers_silver))
print("Payments:", len(payments_silver))


print("\nGOLD DATA")
print("-" * 40)
print("Fact Flights:", len(fact_flights))
print("Fact Bookings:", len(fact_bookings))
print("Fact Payments:", len(fact_payments))
print("Dim Route:", len(dim_route))
print("Dim Airline:", len(dim_airline))
print("Dim Date:", len(dim_date))
print("Dim Passenger:", len(dim_passenger))


print("\nDATA QUALITY")
print("-" * 40)
print("Invalid Flight IDs:", len(invalid_flight_ids))
print(
    "Flight duplicates removed:",
    flight_duplicates_removed
)
print(
    "Duration corrections:",
    duration_corrections
)
print(
    "Overnight flights:",
    fact_flights["is_overnight"].sum()
)
print(
    "Extreme duration anomalies:",
    fact_flights["duration_anomaly"].sum()
)
print(
    "Valid payments:",
    fact_payments["payment_valid"].sum()
)
print(
    "Invalid/missing payments:",
    (~fact_payments["payment_valid"]).sum()
)


print("\n" + "=" * 70)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)

ASG AIRLINES DATA ENGINEERING PIPELINE
FINAL SUMMARY

RAW DATA
----------------------------------------
Flights: 1020
Bookings: 1000
Passengers: 1039
Payments: 1000

SILVER DATA
----------------------------------------
Flights: 1004
Bookings: 1000
Passengers: 1039
Payments: 1000

GOLD DATA
----------------------------------------
Fact Flights: 1004
Fact Bookings: 1000
Fact Payments: 1000
Dim Route: 30
Dim Airline: 6
Dim Date: 369
Dim Passenger: 1039

DATA QUALITY
----------------------------------------
Invalid Flight IDs: 0
Flight duplicates removed: 16
Duration corrections: 1
Overnight flights: 122
Extreme duration anomalies: 0
Valid payments: 922
Invalid/missing payments: 78

PIPELINE COMPLETED SUCCESSFULLY
